# 04 - Fine-tune XLM-RoBERTa

Fine-tune `xlm-roberta-base` for Bangla toxic comment classification.

**Run this notebook on Kaggle or Colab with GPU enabled.**

In [ ]:
# Google Colab setup (does nothing when run locally).
# Clones the repo into Google Drive so results/ survive disconnects, then installs dependencies.
# Before the first run: Runtime > Change runtime type > T4 GPU, and upload train.csv (see below).
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/Set_A'
    if not os.path.isdir(REPO_DIR):
        !git clone -b bangal_profanity https://github.com/mehediparvez/Set_A.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}/BanglaProfanityDetection/notebooks
    !pip install -q transformers datasets accelerate scikit-learn better-profanity groq google-generativeai pyyaml seaborn tabulate
    if not os.path.exists('../data/train.csv'):
        print(f'Missing dataset: upload train.csv to {REPO_DIR}/BanglaProfanityDetection/data/train.csv')

In [ ]:
import sys
import time
import torch
sys.path.append('..')

from src.dataset import load_dataset
from src.train import train_with_multiple_runs, train_learning_curve
from src.evaluate import evaluate_model, get_classification_report, aggregate_runs
from src.model import load_trained_model
from src.results import build_evaluation, save_evaluation
from src.utils import load_config, save_metrics, ensure_dirs

config = load_config('../configs/config.yaml')
model_config = config['models']['xlmr']
MODEL_NAME = model_config['name']
RESULTS_KEY = 'xlmr'  # results/<key>/ is read by notebook 06 and main.py

ensure_dirs('../results', '../plots', '../saved_models')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Model: {MODEL_NAME}')

## Load Dataset

In [ ]:
DATA_PATH = '../' + config['data']['train_path']

train_texts, test_texts, train_labels, test_labels = load_dataset(
    DATA_PATH,
    text_column=config['data']['text_column'],
    label_column=config['data']['label_column'],
    test_size=config['data']['test_size'],
    random_seed=config['data']['random_seed'],
)
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

## Train (3 Runs with Error Bars)

In [ ]:
all_metrics = train_with_multiple_runs(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    output_dir='../results',
    num_runs=config['training']['num_runs'],
    seed=config['data']['random_seed'],
    val_size=config['training']['val_size'],
)

In [ ]:
summary = aggregate_runs(all_metrics)
print('\n=== XLM-RoBERTa Results (3 runs) ===')
for key, stats in summary.items():
    if 'eval_' in key:
        print(f"  {key}: {stats['mean']:.4f} +/- {stats['std']:.4f}")

save_metrics(summary, f'../results/{RESULTS_KEY}/summary.json')
save_metrics(all_metrics, f'../results/{RESULTS_KEY}/all_runs.json')

## Learning Curve (25%, 50%, 100%)

In [ ]:
lc_results = train_learning_curve(
    MODEL_NAME,
    train_texts, test_texts,
    train_labels, test_labels,
    model_config,
    fractions=[0.25, 0.5, 1.0],
    output_dir='../results',
    seed=config['data']['random_seed'],
    val_size=config['training']['val_size'],
)

print('\n=== Learning Curve ===')
for r in lc_results:
    print(f"  {r['fraction']*100:.0f}% ({r['num_samples']} samples): "
          f"Acc={r['eval_accuracy']:.4f}, F1={r['eval_f1']:.4f}")

save_metrics(lc_results, f'../results/{RESULTS_KEY}/learning_curve.json')

## Evaluate Best Model

In [ ]:
best_model_dir = f'../results/{MODEL_NAME.split("/")[-1]}/run_1/best_model'
model, tokenizer = load_trained_model(best_model_dir)
if torch.cuda.is_available():
    model = model.cuda()

start_time = time.time()
_, preds = evaluate_model(model, tokenizer, test_texts, test_labels, max_length=model_config['max_length'])
evaluation = build_evaluation(test_labels, preds, time.time() - start_time)
save_evaluation(evaluation, '../results', RESULTS_KEY)

print('\n=== Best Model Evaluation ===')
for k, v in evaluation['metrics'].items():
    print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
print(get_classification_report(test_labels, preds))

print('\n=== Confusion Matrix ===')
print(evaluation['confusion_matrix'])

In [ ]:
# Colab: make sure everything written to Google Drive is actually uploaded before the runtime ends
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.flush_and_unmount()
    print('All results saved to Google Drive.')